# Patient-Grouped Stratified Data Split

This notebook creates the final train, validation, and test splits for the diabetes readmission dataset. We formulate the target as a binary classification problem: `readmitted = 0` represents no recorded inpatient readmission, while `readmitted = 1` represents any recorded inpatient readmission, combining both `<30` and `>30` days.

Because the dataset contains multiple encounters for some patients, the split is performed at the patient level using `patient_nbr`. This prevents encounters from the same patient appearing in more than one dataset split, reducing patient-level information leakage. At the same time, stratification is used to preserve a similar target-class distribution across the train, validation, and test sets.

---

Before splitting, encounters with discharge dispositions indicating death or hospice care are excluded, since later readmission is not a comparable outcome for those patients.




#### **Import Libraries and Custom Functions**

In [1]:
import pandas as pd
from pathlib import Path

from src.utils.data_split.functions import (split_data_by_patient_stratified,
                                            create_y_class_report,
                                            export_split_data_to_csv,
                                            binarize_readmitted
                                            )

#### **Import the dataset:**

In [2]:
# initializing the paths
PATH = Path("../../data/source/diabetic_data.csv")
# used for exporting the splitted datasets
SPLITTED_PATH = Path("../../data/interim/split_kf_group_stratified")
 #reading the datasets
df = pd.read_csv(PATH)
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In this project we want to formulate readmission prediction as a binary classification task where the positive class represents any recorded inpatient readmission after the index encounter, combining both <30 and >30 readmissions. Because multiple encounters can belong to the same patient, we split the data at the patient level to prevent information leakage across train, validation, and test sets.

**As mentioned above, we want to predict whether a hospital encounter will be followed by any recorded inpatient readmission, regardless of whether it happens before or after 30 days.**

In other words we binarize the target variable 'readmitted' as follows:
- 'No readmission' is mapped to 0,
- '>30' days is mapped to 1,
- and '<30' days is mapped to 1.

In [3]:
df = binarize_readmitted(df)
df[['encounter_id', 'readmitted']].head()

,encounter_id,readmitted
0,2278392,0
1,149190,1
2,64410,0
3,500364,0
4,16680,0


### Applying the Stratified Patient-Grouped Split

**Applying the data split function**

**IMPORTANT**: Keep in mind that we exclude encounters where the discharge disposition indicated death or hospice care, because subsequent hospital readmission is not a comparable outcome for these patients.

In [4]:
X_train_plus_val, X_train_mini, X_val, X_test, \
    y_train_plus_val, y_train_mini, y_val, y_test = \
            split_data_by_patient_stratified(df, random_state= 365)

In [17]:
# draft check:
temp_xtrain = X_train_mini['patient_nbr'].unique().tolist()
temp_xval = X_val['patient_nbr'].unique().tolist()
# check if there are any patients in both train and val
set(temp_xtrain).intersection(set(temp_xval))

pd.concat([X_train_mini, X_val, X_test]).shape

(99343, 49)

**Brief report of target class counts and percentages for each dataset**:

In [5]:
y_class_report = create_y_class_report(y_train_mini, y_val, y_test)
y_class_report

,Dataset,Class,Count,Percentage,Total Rows
0,Train,1,28090,47.13,59605
1,Train,0,31515,52.87,59605
2,Validation,1,9363,47.12,19869
3,Validation,0,10506,52.88,19869
4,Test,1,9363,47.12,19869
5,Test,0,10506,52.88,19869


**Export datasets to ../data/interim/folder_name/**

In [6]:
split_file_paths = export_split_data_to_csv(
    X_train_plus_val, X_train_mini, X_val, X_test,
    y_train_plus_val, y_train_mini, y_val, y_test,
    output_folder= SPLITTED_PATH
)

Exported 8 split CSV files to: ../../data/interim/split_kf_group_stratified
- X_train_plus_val: ../../data/interim/split_kf_group_stratified/X_train_plus_val.csv
- X_train_mini: ../../data/interim/split_kf_group_stratified/X_train_mini.csv
- X_val: ../../data/interim/split_kf_group_stratified/X_val.csv
- X_test: ../../data/interim/split_kf_group_stratified/X_test.csv
- y_train_plus_val: ../../data/interim/split_kf_group_stratified/y_train_plus_val.csv
- y_train_mini: ../../data/interim/split_kf_group_stratified/y_train_mini.csv
- y_val: ../../data/interim/split_kf_group_stratified/y_val.csv
- y_test: ../../data/interim/split_kf_group_stratified/y_test.csv
